# Financial Performance Analytics & Forecasting

An end-to-end machine-learning study that uses historical firm/bank financial indicators to forecast next-year Income from financial services.

Dataset: 213 firms/banks across seven yearly blocks.

Primary methodology: data-quality audit → predictor-only feature engineering → expanding-window walk-forward validation → model comparison using RMSE, MAE and R².

IMPORTANT: the supplied dataset contains an exact duplicate final target block. The duplicated final transition is excluded from the primary model score so that performance is not overstated.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
FEATURES = [
    "Other income",
    "Profit after tax",
    "Total capital",
    "Reserves and funds",
    "Deposits (accepted by commercial banks)",
    "Current liabilities & provisions",
]
TARGET = "Income from financial services"


## 1. Load and audit the dataset

Before modelling, we inspect dimensions, missing values, duplicate rows and the repeated yearly structure.


In [ ]:
DATA_PATH = Path("data/7_year.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/content/7_year.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))
print(f"Exact duplicate rows: {df.duplicated().sum()}")


## 2. Build year-wise blocks

Each year contains the same six predictor variables plus the target. We convert the wide dataset into six one-year-ahead forecasting transitions.


In [ ]:
years = []
all_year_features = FEATURES + [TARGET]

for year in range(7):
    suffix = "" if year == 0 else f".{year}"
    cols = [c + suffix for c in all_year_features]
    block = df[["Company Name"] + cols].copy()
    block.columns = ["Company Name"] + all_year_features
    block["Year"] = year + 1
    years.append(block)

print(f"Year blocks: {len(years)}")
print(f"Firms per block: {len(years[0])}")


## 3. Data-quality check: duplicated final target

The final target block is audited explicitly. If it is an exact duplicate, we do not use that transition as the primary evidence for model quality.


In [ ]:
target_y6 = years[5][TARGET].to_numpy()
target_y7 = years[6][TARGET].to_numpy()

audit = pd.Series({
    "Exact duplicate": np.array_equal(target_y6, target_y7),
    "Matching share": np.mean(np.isclose(target_y6, target_y7)),
    "Maximum absolute difference": np.max(np.abs(target_y6 - target_y7)),
}, name="Value")

display(audit.to_frame())


## 4. Create one-year-ahead forecasting transitions

For transition t → t+1, only information available in year t is used as the predictor. The target is the next year's Income from financial services.


In [ ]:
transitions = []

for t in range(6):
    current = years[t]
    nxt = years[t + 1]
    block = current[FEATURES].copy()
    block["Company Name"] = current["Company Name"].values
    block["Forecast From Year"] = t + 1
    block["Target Year"] = t + 2
    block["Target"] = nxt[TARGET].values
    transitions.append(block)

panel = pd.concat(transitions, ignore_index=True)
display(panel.head())
print(f"Forecasting observations: {len(panel)}")


## 5. Predictor-only feature engineering

Financial ratios are calculated only from variables available at forecast time. The future target is never used to create a predictor, preventing target leakage.


In [ ]:
def add_financial_features(data):
    out = data.copy()
    eps = 1e-9
    capital = out["Total capital"].abs() + eps
    out["Profit_to_Capital"] = out["Profit after tax"] / capital
    out["Reserves_to_Capital"] = out["Reserves and funds"] / capital
    out["Liabilities_to_Capital"] = out["Current liabilities & provisions"] / capital
    out["Deposits_to_Capital"] = out["Deposits (accepted by commercial banks)"] / capital
    out["OtherIncome_to_Capital"] = out["Other income"] / capital
    return out.replace([np.inf, -np.inf], np.nan).fillna(0.0)

model_data = add_financial_features(panel)
ENGINEERED = [
    "Profit_to_Capital",
    "Reserves_to_Capital",
    "Liabilities_to_Capital",
    "Deposits_to_Capital",
    "OtherIncome_to_Capital",
]


## 6. Establish a naive baseline

A machine-learning model should beat a simple reference model. We use the target observed in the forecast-origin year as a persistence baseline.


In [ ]:
baseline_rows = []

for target_year in [3, 4, 5, 6]:
    test = model_data[model_data["Target Year"] == target_year]
    origin_year = target_year - 1
    origin_target = years[origin_year - 1][TARGET].to_numpy()

    baseline_rows.append({
        "Test transition": f"Year {origin_year} -> Year {target_year}",
        "RMSE": mean_squared_error(test["Target"], origin_target) ** 0.5,
        "MAE": mean_absolute_error(test["Target"], origin_target),
        "R2": r2_score(test["Target"], origin_target),
    })

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results)


## 7. Model definitions

The comparison covers linear, regularized, kernel and tree-based regressors. Scaling is applied inside pipelines for models that benefit from it.


In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "Ridge": Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    "Lasso": Pipeline([
        ("scale", StandardScaler()),
        ("model", Lasso(alpha=0.001, max_iter=20000)),
    ]),
    "SVR (RBF)": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.01)),
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=400, min_samples_leaf=2, max_features=1.0,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.03, max_depth=2,
        loss="huber", random_state=RANDOM_STATE,
    ),
}

MODEL_FEATURES = FEATURES + ENGINEERED


## 8. Expanding-window walk-forward validation

We evaluate transitions from Year 2→3 through Year 5→6. The duplicated Year 6→7 target transition is excluded from the primary score. At every step, only earlier target years are available for training.


In [ ]:
results = []

for test_target_year in [3, 4, 5, 6]:
    train = model_data[model_data["Target Year"] < test_target_year]
    test = model_data[model_data["Target Year"] == test_target_year]

    for name, model in models.items():
        fitted = clone(model)
        fitted.fit(train[MODEL_FEATURES], train["Target"])
        pred = fitted.predict(test[MODEL_FEATURES])

        results.append({
            "Test transition": f"Year {test_target_year - 1} -> Year {test_target_year}",
            "Model": name,
            "RMSE": mean_squared_error(test["Target"], pred) ** 0.5,
            "MAE": mean_absolute_error(test["Target"], pred),
            "R2": r2_score(test["Target"], pred),
        })

results_df = pd.DataFrame(results)
display(results_df)


## 9. Aggregate model comparison

Mean metrics across the independent walk-forward transitions provide a more robust summary than a single final-year score.


In [ ]:
summary = (
    results_df.groupby("Model")[["RMSE", "MAE", "R2"]]
    .mean()
    .sort_values("RMSE")
)

display(summary)


In [ ]:
ax = summary["RMSE"].plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Mean RMSE")
ax.set_title("Walk-forward Model Comparison")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 10. Best-model diagnostic: predicted vs actual

The model with the lowest mean walk-forward RMSE is inspected across all evaluated transitions. This is descriptive model analysis, not a guarantee of future performance.


In [ ]:
best_model_name = summary.index[0]
best_model = models[best_model_name]
pred_rows = []

for test_target_year in [3, 4, 5, 6]:
    train = model_data[model_data["Target Year"] < test_target_year]
    test = model_data[model_data["Target Year"] == test_target_year]

    fitted = clone(best_model)
    fitted.fit(train[MODEL_FEATURES], train["Target"])
    pred = fitted.predict(test[MODEL_FEATURES])

    pred_rows.append(pd.DataFrame({
        "Transition": f"Year {test_target_year - 1} -> Year {test_target_year}",
        "Actual": test["Target"].values,
        "Predicted": pred,
    }))

predictions = pd.concat(pred_rows, ignore_index=True)
display(predictions.head(10))


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(predictions["Actual"], predictions["Predicted"], alpha=0.6)

lims = [
    predictions[["Actual", "Predicted"]].min().min(),
    predictions[["Actual", "Predicted"]].max().max(),
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(f"Predicted vs Actual — {best_model_name}")
plt.tight_layout()
plt.show()


## 11. Final interpretation

The improved experiment now has a clear separation between data validation, feature engineering, model training and evaluation.

### What to report

- Compare models using RMSE, MAE and R², rather than RMSE alone.
- Compare against the naive persistence baseline.
- Explain that chronological validation reduces future-information leakage.
- Disclose the duplicated final target block.
- Do not claim that the model guarantees future financial performance.

### Application layer

app.py provides the Streamlit interface for exploring firms and model predictions. This notebook is the reproducible research layer; the dashboard is the presentation/application layer.
